# PathRAG Pipeline — Interactive Step-by-Step Walkthrough

This notebook lets you explore each stage of the PathRAG pipeline interactively.
Each cell is self-contained so you can run them in any order.

| Step | Description | LLM Required |
|------|-------------|:------------:|
| 1 | **Chunking** — Token-based document splitting | No |
| 2 | **Path Finding & Pruning** — Multi-hop path discovery on a graph | No |
| 3 | **Custom KG Insert** — Manual knowledge graph insertion | No |
| 4 | **Graph Storage Operations** — Direct CRUD on NetworkXStorage | No |
| 5 | **Entity Deletion** — Remove entities from the KG | No |
| 6 | **Full Indexing Pipeline** — Document → Chunking → Entity Extraction → KG | Yes |
| 7 | **Full Query Pipeline** — Keyword Extraction → Context Retrieval → Path Finding → Response | Yes |

**Prerequisites:**
- `pip install -e .` (install PathRAG package)
- Copy `examples/.env.example` to `examples/.env` and set your API key

## Setup — Load environment variables and define sample data

In [ ]:
import os
import json
import shutil
import tempfile

import networkx as nx
import numpy as np
from dotenv import load_dotenv, find_dotenv

# Load .env file automatically
load_dotenv(find_dotenv())


def _get_llm_config():
    """Resolve LLM / embedding config from environment variables.

    Returns:
        (llm_model, embedding_model, embedding_dim, tiktoken_model)
        tiktoken_model is derived from LLM_MODEL_NAME and passed to
        tiktoken for token counting.  Falls back to cl100k_base encoding
        when the model is not recognized by tiktoken (e.g. Gemini,
        Bedrock, Ollama models).
    """
    if os.environ.get("GEMINI_API_KEY"):
        default_llm = "gemini/gemini-2.5-flash"
        default_embed = "gemini/gemini-embedding-001"
        default_dim = 3072
    elif os.environ.get("OPENAI_API_KEY"):
        default_llm = "gpt-4o-mini"
        default_embed = "text-embedding-3-small"
        default_dim = 1536
    else:
        return None, None, None, None

    llm_model = os.environ.get("LLM_MODEL_NAME", default_llm)
    embedding_model = os.environ.get("EMBEDDING_MODEL_NAME", default_embed)
    embedding_dim = int(os.environ.get("EMBEDDING_DIM", default_dim))

    # Use LLM_MODEL_NAME as the tiktoken model name.
    # tiktoken recognises OpenAI model names (e.g. gpt-4o-mini) directly.
    # For non-OpenAI models (e.g. gemini/gemini-2.5-flash),
    # _get_tiktoken_encoder() falls back to cl100k_base automatically.
    tiktoken_model = llm_model

    print(f"LLM Model      : {llm_model}")
    print(f"Embedding Model: {embedding_model} (dim={embedding_dim})")
    print(f"Tiktoken Model : {tiktoken_model}")
    return llm_model, embedding_model, embedding_dim, tiktoken_model


SAMPLE_DOCUMENTS = [
    """
    Apple Inc. is an American multinational technology company headquartered in
    Cupertino, California. Apple was founded on April 1, 1976, by Steve Jobs,
    Steve Wozniak, and Ronald Wayne. The company designs, develops, and sells
    consumer electronics, computer software, and online services.
    Tim Cook has been the CEO of Apple since August 2011, succeeding Steve Jobs.
    Under Cook's leadership, Apple launched Apple Watch, AirPods, and Apple Vision Pro.
    """,
    """
    Steve Jobs was an American business magnate, inventor, and investor. He was
    the co-founder, chairman, and CEO of Apple Inc. Jobs also co-founded and
    served as the chairman of Pixar Animation Studios. He was a board member at
    The Walt Disney Company following the acquisition of Pixar by Disney.
    Jobs is widely recognized as a pioneer of the personal computer revolution
    and for his influential career in the computer and consumer electronics fields.
    """,
    """
    Google LLC is an American multinational corporation and technology company
    focusing on online advertising, search engine technology, cloud computing,
    and artificial intelligence. Sundar Pichai has been the CEO of Google since
    October 2015 and of its parent company Alphabet Inc. since December 2019.
    Google was founded on September 4, 1998, by Larry Page and Sergey Brin
    while they were Ph.D. students at Stanford University in California.
    """,
]

print("Setup complete. Sample documents loaded.")
print(f"API key detected: {'GEMINI' if os.environ.get('GEMINI_API_KEY') else 'OPENAI' if os.environ.get('OPENAI_API_KEY') else 'NONE'}")

---
## Step 1: Chunking

Split documents into overlapping token-sized chunks. This is the first stage of the indexing pipeline.

Try changing `max_token_size` and `overlap_token_size` to see how they affect the output.

In [ ]:
from PathRAG.operate import chunking_by_token_size

# Adjust these parameters to experiment
MAX_TOKEN_SIZE = 200
OVERLAP_TOKEN_SIZE = 50

# Use LLM_MODEL_NAME for tiktoken token counting.
# Falls back to cl100k_base for non-OpenAI models (e.g. Gemini).
tiktoken_model = os.environ.get("LLM_MODEL_NAME", "gpt-4o")

for i, doc in enumerate(SAMPLE_DOCUMENTS):
    chunks = chunking_by_token_size(
        doc.strip(),
        overlap_token_size=OVERLAP_TOKEN_SIZE,
        max_token_size=MAX_TOKEN_SIZE,
        tiktoken_model=tiktoken_model,
    )
    print(f"--- Document {i+1} ---")
    print(f"  Original length : {len(doc.strip())} chars")
    print(f"  Chunks created  : {len(chunks)}")
    for j, chunk in enumerate(chunks):
        print(f"  Chunk {j}: tokens={chunk['tokens']}, order={chunk['chunk_order_index']}")
        print(f"    {chunk['content'][:100]}...")
    print()

---
## Step 2: Path Finding & Pruning (no LLM required)

This is PathRAG's core innovation. Given a set of target entities retrieved by vector search,
the algorithm:
1. Discovers all paths (1-hop, 2-hop, 3-hop) between them using DFS
2. Weights paths by traversal frequency using BFS
3. Prunes weak connections below a threshold

Try modifying `target_nodes`, `threshold`, and `alpha` to see how they affect path selection.

In [ ]:
from PathRAG.operate import find_paths_and_edges_with_stats, bfs_weighted_paths

# Build a sample knowledge graph
G = nx.Graph()
graph_edges = [
    ("STEVE JOBS", "APPLE"),
    ("STEVE JOBS", "PIXAR"),
    ("APPLE", "TIM COOK"),
    ("APPLE", "CUPERTINO"),
    ("PIXAR", "DISNEY"),
    ("DISNEY", "ENTERTAINMENT"),
    ("TIM COOK", "APPLE WATCH"),
    ("GOOGLE", "SUNDAR PICHAI"),
    ("GOOGLE", "ALPHABET"),
    ("GOOGLE", "STANFORD UNIVERSITY"),
    ("LARRY PAGE", "GOOGLE"),
    ("SERGEY BRIN", "GOOGLE"),
    ("LARRY PAGE", "STANFORD UNIVERSITY"),
]
G.add_edges_from(graph_edges)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Simulate entities retrieved by vector search — try changing these!
target_nodes = ["STEVE JOBS", "TIM COOK", "APPLE", "PIXAR"]

print(f"Target nodes: {target_nodes}\n")

# DFS-based path discovery
result, path_stats, one_hop, two_hop, three_hop = await find_paths_and_edges_with_stats(G, target_nodes)

print("--- Path Statistics ---")
for hop, count in path_stats.items():
    print(f"  {hop}: {count}")

for label, paths in [("1-hop", one_hop), ("2-hop", two_hop), ("3-hop", three_hop)]:
    print(f"\n--- {label} Paths ---")
    for p in paths[:5]:
        print(f"  {' -> '.join(p)}")

In [ ]:
# BFS-based path weighting and pruning
# Adjust these to see how pruning behaviour changes
THRESHOLD = 0.3   # Minimum edge weight to keep exploring
ALPHA = 0.8       # Decay factor per hop

print(f"Pruning parameters: threshold={THRESHOLD}, alpha={ALPHA}\n")

for (src, tgt), data in list(result.items())[:4]:
    paths = data["paths"]
    if not paths:
        continue
    weighted = bfs_weighted_paths(G, paths, src, tgt, THRESHOLD, ALPHA)
    print(f"{src} -> {tgt}: ({len(paths)} path(s))")
    for path, weight in sorted(weighted, key=lambda x: x[1], reverse=True)[:5]:
        status = "KEEP " if weight > THRESHOLD else "PRUNE"
        print(f"  [{status}] weight={weight:.3f}  {' -> '.join(path)}")
    print()

---
## Step 3: Custom KG (Knowledge Graph) Insert (no LLM required)

Insert a hand-crafted knowledge graph into PathRAG storage.
This uses a dummy embedding function so no API key is needed.

In [ ]:
from PathRAG import PathRAG, QueryParam
from PathRAG.utils import EmbeddingFunc

async def dummy_embedding(texts: list[str]) -> np.ndarray:
    return np.random.randn(len(texts), 384).astype(np.float32)

work_dir = tempfile.mkdtemp(prefix="pathrag_nb_custom_kg_")

rag = PathRAG(
    working_dir=work_dir,
    embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=8192, func=dummy_embedding),
    embedding_dim=384,
)

custom_kg = {
    "chunks": [
        {"content": "Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.", "source_id": "chunk-apple-founding"},
        {"content": "Steve Jobs co-founded Pixar and served as its chairman until Disney acquired it.", "source_id": "chunk-jobs-pixar"},
    ],
    "entities": [
        {"entity_name": "Apple Inc.", "entity_type": "ORGANIZATION", "description": "American multinational technology company", "source_id": "chunk-apple-founding"},
        {"entity_name": "Steve Jobs", "entity_type": "PERSON", "description": "Co-founder and former CEO of Apple Inc.", "source_id": "chunk-apple-founding"},
        {"entity_name": "Pixar", "entity_type": "ORGANIZATION", "description": "Animation studio co-founded by Steve Jobs", "source_id": "chunk-jobs-pixar"},
    ],
    "relationships": [
        {"src_id": "Steve Jobs", "tgt_id": "Apple Inc.", "description": "Steve Jobs co-founded Apple Inc. in 1976", "keywords": "founding, co-founder, leadership", "weight": 3.0, "source_id": "chunk-apple-founding"},
        {"src_id": "Steve Jobs", "tgt_id": "Pixar", "description": "Steve Jobs co-founded Pixar Animation Studios", "keywords": "co-founder, animation, chairman", "weight": 2.0, "source_id": "chunk-jobs-pixar"},
    ],
}

await rag.ainsert_custom_kg(custom_kg)

# Inspect results
graph = rag.chunk_entity_relation_graph
nodes = await graph.nodes()
edges_list = await graph.edges()

print(f"Inserted nodes: {len(nodes)}, edges: {len(edges_list)}\n")

print("--- Nodes ---")
for name in nodes:
    data = await graph.get_node(name)
    print(f"  {name}: type={data.get('entity_type')}, desc={data.get('description', '')[:60]}")

print("\n--- Edges ---")
for src, tgt in edges_list:
    data = await graph.get_edge(src, tgt)
    print(f"  {src} -> {tgt}: weight={data.get('weight')}, keywords={data.get('keywords')}")

# Cleanup
shutil.rmtree(work_dir, ignore_errors=True)

---
## Step 4: Graph Storage Operations (no LLM required)

Directly manipulate the `NetworkXStorage` graph backend — insert nodes/edges,
query properties, and persist to disk.

In [ ]:
from PathRAG.storage import NetworkXStorage
from PathRAG.utils import EmbeddingFunc

work_dir = tempfile.mkdtemp(prefix="pathrag_nb_graph_")

async def dummy_embedding(texts: list[str]) -> np.ndarray:
    return np.random.randn(len(texts), 384).astype(np.float32)

graph = NetworkXStorage(
    namespace="test_graph",
    global_config={"working_dir": work_dir},
    embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=8192, func=dummy_embedding),
)

# Insert nodes
await graph.upsert_node('"APPLE"', node_data={"entity_type": "ORGANIZATION", "description": "American multinational technology company", "source_id": "chunk-001"})
await graph.upsert_node('"STEVE JOBS"', node_data={"entity_type": "PERSON", "description": "Co-founder of Apple Inc.", "source_id": "chunk-001"})
await graph.upsert_node('"TIM COOK"', node_data={"entity_type": "PERSON", "description": "Current CEO of Apple Inc.", "source_id": "chunk-002"})

# Insert edges
await graph.upsert_edge('"STEVE JOBS"', '"APPLE"', edge_data={"weight": 3.0, "description": "Co-founded Apple in 1976", "keywords": "founding, co-founder", "source_id": "chunk-001"})
await graph.upsert_edge('"TIM COOK"', '"APPLE"', edge_data={"weight": 2.5, "description": "CEO of Apple since 2011", "keywords": "CEO, leadership", "source_id": "chunk-002"})

# Query
print(f"Node count: {len(await graph.nodes())}")
print(f"Edge count: {len(await graph.edges())}")

node = await graph.get_node('"APPLE"')
print(f'\n"APPLE" node:\n{json.dumps(node, indent=2)}')

edge = await graph.get_edge('"STEVE JOBS"', '"APPLE"')
print(f'\n"STEVE JOBS" -> "APPLE" edge:\n{json.dumps(edge, indent=2)}')

degree = await graph.node_degree('"APPLE"')
print(f'\n"APPLE" degree: {degree}')

# Persist and cleanup
await graph.index_done_callback()
print(f"\nGraph saved to: {work_dir}")
shutil.rmtree(work_dir, ignore_errors=True)

---
## Step 5: Entity Deletion (no LLM required)

Insert a small KG and then delete an entity. Observe how nodes, edges, and
vector DB entries are cleaned up.

In [ ]:
work_dir = tempfile.mkdtemp(prefix="pathrag_nb_delete_")

async def dummy_embedding(texts: list[str]) -> np.ndarray:
    return np.random.randn(len(texts), 384).astype(np.float32)

rag = PathRAG(
    working_dir=work_dir,
    embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=8192, func=dummy_embedding),
    embedding_dim=384,
)

await rag.ainsert_custom_kg({
    "chunks": [{"content": "Test chunk.", "source_id": "src-1"}],
    "entities": [
        {"entity_name": "EntityA", "entity_type": "TEST", "description": "Test entity A", "source_id": "src-1"},
        {"entity_name": "EntityB", "entity_type": "TEST", "description": "Test entity B", "source_id": "src-1"},
    ],
    "relationships": [
        {"src_id": "EntityA", "tgt_id": "EntityB", "description": "A relates to B", "keywords": "test", "weight": 1.0, "source_id": "src-1"},
    ],
})

graph = rag.chunk_entity_relation_graph
print(f"Before deletion — nodes: {len(await graph.nodes())}, edges: {len(await graph.edges())}")
print(f"  Nodes: {list(await graph.nodes())}")

# Delete EntityA
await rag.adelete_by_entity("EntityA")

print(f"\nAfter deletion  — nodes: {len(await graph.nodes())}, edges: {len(await graph.edges())}")
print(f"  Nodes: {list(await graph.nodes())}")

shutil.rmtree(work_dir, ignore_errors=True)

---
## Step 6: Full Indexing Pipeline (LLM required)

Run the complete indexing pipeline on a single document:
**Document → Chunking → LLM Entity Extraction → Knowledge Graph Construction**

> Requires `GEMINI_API_KEY` or `OPENAI_API_KEY` in your `.env` file.

In [ ]:
llm_model, embedding_model, embedding_dim, tiktoken_model = _get_llm_config()
assert llm_model is not None, "No API key found. Set GEMINI_API_KEY or OPENAI_API_KEY in .env"

work_dir = tempfile.mkdtemp(prefix="pathrag_nb_index_")

rag = PathRAG(
    working_dir=work_dir,
    llm_model_name=llm_model,
    embedding_model_name=embedding_model,
    embedding_dim=embedding_dim,
    tiktoken_model_name=tiktoken_model,
    chunk_token_size=200,
    chunk_overlap_token_size=50,
)

print("Indexing first document...")
await rag.ainsert(SAMPLE_DOCUMENTS[0])
print("Done!\n")

# Inspect extracted KG
graph = rag.chunk_entity_relation_graph
nodes = await graph.nodes()
edges_list = await graph.edges()
print(f"Extracted entities: {len(nodes)}")
print(f"Extracted relationships: {len(edges_list)}")

print("\n--- Entities ---")
for name in list(nodes)[:10]:
    data = await graph.get_node(name)
    if data:
        print(f"  {name}: type={data.get('entity_type', 'N/A')}")

print("\n--- Relationships ---")
for src, tgt in list(edges_list)[:10]:
    data = await graph.get_edge(src, tgt)
    if data:
        print(f"  {src} --[{data.get('keywords', 'N/A')}]--> {tgt}")

# Keep work_dir for the next step (query pipeline)
print(f"\nWorking directory kept at: {work_dir}")

---
## Step 7: Full Query Pipeline (LLM required)

Run the end-to-end query pipeline on all three sample documents:
**Keyword Extraction → Dual Context Retrieval → Path Finding → LLM Response**

This step builds a fresh index from all documents, then demonstrates three
query modes:
1. **Context only** — see the raw context retrieved from the KG
2. **Prompt inspection** — see the final prompt sent to the LLM
3. **Full response** — get the LLM-generated answer

> Try your own questions in the last cell!

In [ ]:
llm_model, embedding_model, embedding_dim, tiktoken_model = _get_llm_config()
assert llm_model is not None, "No API key found. Set GEMINI_API_KEY or OPENAI_API_KEY in .env"

query_work_dir = tempfile.mkdtemp(prefix="pathrag_nb_query_")

rag = PathRAG(
    working_dir=query_work_dir,
    llm_model_name=llm_model,
    embedding_model_name=embedding_model,
    embedding_dim=embedding_dim,
    tiktoken_model_name=tiktoken_model,
    chunk_token_size=300,
    chunk_overlap_token_size=50,
)

print(f"Indexing {len(SAMPLE_DOCUMENTS)} documents...")
await rag.ainsert(SAMPLE_DOCUMENTS)
print("Indexing complete!")

### 7-1: Context retrieval only

See what context the PathRAG retriever builds from the KG — without generating a final LLM response.

In [ ]:
query = "What is the relationship between Steve Jobs and Apple?"

context = await rag.aquery(
    query,
    param=QueryParam(mode="hybrid", only_need_context=True, top_k=10),
)
print(f"Query  : {query}")
print(f"Context: {len(context)} chars\n")
print(context[:1000])
print("..." if len(context) > 1000 else "")

### 7-2: Final prompt inspection

See the full system prompt (context + instructions) that gets sent to the LLM.

In [ ]:
prompt = await rag.aquery(
    query,
    param=QueryParam(mode="hybrid", only_need_prompt=True, top_k=10),
)
print(f"Prompt length: {len(prompt)} chars\n")
print(prompt[:1500])
print("..." if len(prompt) > 1500 else "")

### 7-3: Full response generation

Ask questions and get complete LLM-generated answers backed by the knowledge graph.

In [ ]:
queries = [
    "What is the relationship between Steve Jobs and Apple?",
    "Who founded Google and what is their background?",
    "Compare the leadership of Apple and Google.",
]

for q in queries:
    print(f"Q: {q}")
    response = await rag.aquery(q, param=QueryParam(mode="hybrid", top_k=10))
    print(f"A: {response}\n")
    print("-" * 70)

### 7-4: Try your own question!

Edit the `my_question` variable below and run the cell.

In [ ]:
# Edit this question and re-run the cell!
my_question = "What role did Steve Jobs play at Pixar?"

response = await rag.aquery(my_question, param=QueryParam(mode="hybrid", top_k=10))
print(f"Q: {my_question}\n")
print(f"A: {response}")

---
## Cleanup

Run this cell to remove the temporary working directory created by the query pipeline.

In [ ]:
# Remove temporary directories
for d in [d for d in dir() if d.endswith("work_dir") or d.endswith("_work_dir")]:
    path = eval(d)
    if isinstance(path, str) and os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"Removed: {path}")

print("Cleanup complete.")